# Lab 54: The Crossed Confusion

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 6, *The Embodied Avatar*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-54.ipynb)

**What you will do:** measure how often you correctly identify which hand was tapped with hands crossed versus uncrossed, check the pattern against the book's related published timing study, and run a small coordinate-transform model of the same body-to-space mismatch.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 54 you had a friend tap one of your hands while your eyes were closed, first with your
hands uncrossed and then with them crossed over the midline. With hands uncrossed you could say
which hand was tapped almost every time; with hands crossed, errors became much more common and
responses slower. The book explains this with two coordinate systems for touch -- a body-centred
one (which hand?) and a space-centred one (which side of space?) -- that normally agree but
conflict when the hands are crossed.

## Record your results

Run at least 10 trials in each condition (uncrossed, crossed), with your friend tapping hands
in a random order you cannot predict, and record whether you named the tapped hand correctly.

In [ ]:
trials = pd.DataFrame({
    "condition": ["uncrossed"] * 10 + ["crossed"] * 10,
    "trial": list(range(1, 11)) * 2,
    "correct": [1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
                1, 0, 1, 0, 1, 0, 0, 1, 0, 1],   # example data: 1 = correctly named, 0 = error
})
display(trials)
accuracy = trials.groupby("condition")["correct"].mean()
display(accuracy.to_frame("proportion correct"))
accuracy.plot.barh(color=["steelblue", "firebrick"], figsize=(6, 2))
plt.xlim(0, 1); plt.xlabel("proportion correctly identified"); plt.show()

## Compare

The book does not report an error-rate benchmark for this exact naming task, but it does give
a closely related published number: in a temporal-order task, participants needed taps only
34 ms apart to tell them apart with hands uncrossed, but needed a gap of about 124 ms with
hands crossed -- roughly a fourfold increase (Shore et al., 2002). That is a different
measurement (a timing threshold, not naming accuracy), but it comes from the same crossed-hands
conflict.

In [ ]:
uncrossed_acc = accuracy["uncrossed"]
crossed_acc = accuracy["crossed"]
print(f"Uncrossed accuracy: {uncrossed_acc:.0%}, crossed accuracy: {crossed_acc:.0%}")
if crossed_acc < uncrossed_acc:
    print("Your accuracy dropped when your hands were crossed, matching the book's general claim, even")
    print("though your task (naming which hand) and Shore et al.'s (judging which tap came first) are not")
    print("the same measurement -- the published 34ms-to-124ms figure is context for the size of the effect")
    print("in a related task, not a number your own accuracy should match exactly.")
else:
    print("Your accuracy did not drop with crossed hands. A small number of trials is noisy; try more")
    print("trials, and make sure your friend really randomises which hand is tapped.")

## The AI side

The book frames the crossed-hands confusion as a coordinate-transformation problem: converting
a touch location from a body-centred frame ("right hand") into a space-centred frame ("right
side of the room"), a transform that depends on the current configuration of the body. Below,
a simple 2D coordinate transform does the same job, and shows what happens when the transform's
assumed configuration ("arms uncrossed") no longer matches reality.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
def hand_position_in_space(shoulder_x, tap_side, crossed):
    # tap_side is "left" or "right" hand in BODY coordinates.
    # Under normal (uncrossed) posture, the left hand is on the left side of space, and vice versa.
    body_to_space = {"left": -1, "right": 1}
    sign = body_to_space[tap_side]
    if crossed:
        sign *= -1   # crossing swaps which side of space each hand actually occupies
    return shoulder_x + sign * 30   # cm from body midline

for crossed in (False, True):
    print(f"crossed = {crossed}")
    for tap_side in ("left", "right"):
        space_x = hand_position_in_space(0, tap_side, crossed)
        assumed_space_x = hand_position_in_space(0, tap_side, crossed=False)   # the brain's default guess
        match = "MATCH" if space_x == assumed_space_x else "MISMATCH"
        print(f"  {tap_side} hand tapped -> actual position {space_x:+.0f} cm,"
              f" brain's uncrossed-default guess {assumed_space_x:+.0f} cm, {match}")

When the arms are crossed, the model's default, uncrossed-posture assumption gives the wrong
side of space for every tap -- exactly the mismatch the book describes between body-based and
space-based coding. The analogy has limits: this code is handed the correct "crossed" flag and
computes the transform exactly, whereas your brain has to detect the crossed posture from
proprioceptive and visual cues and gradually override an initial default guess, a process that
takes real time and sometimes fails, as your own error rate above shows.

## Questions to think about

1. In the code, hand_position_in_space's assumed_space_x always uses crossed=False, so every crossed-hand tap is a MISMATCH. What would the code need to add to model the way your brain eventually "catches up" and corrects the mismatch after 100-300 ms, as Yamamoto & Kitazawa (2001) describe?
2. Shore et al. (2002) measured a fourfold increase in the required timing gap (34 ms to 124 ms) with crossed hands. Does the drop in your own accuracy (crossed vs uncrossed) look like a similarly large effect, a smaller one, or is accuracy simply the wrong kind of measurement to compare directly to a timing threshold?
3. The book raises augmented-reality systems that must track which physical hand is which as arms cross in the camera's view. Why might this be harder for a computer vision system than for the coordinate-transform code above, which was simply told which hand was tapped rather than having to infer it from an image?
4. The "Beyond the Lab" section argues that the same physical stimulus can mean something different depending on context (crossed vs uncrossed hands). Can you think of an example from language or gesture where the same signal changes meaning depending on some other "configuration", the way a tap's meaning changes with hand posture?

## Challenge

Repeat the crossed-hands test, but this time have your friend vary the gap between two taps (one
on each hand) and ask you which one came first, rather than which hand was tapped. Start with a
large gap (say 300 ms) and shrink it on successive trials, tracking the smallest gap at which you
can still reliably tell which tap came first, separately for crossed and uncrossed hands.

In [ ]:
# Example data: replace with your own estimated thresholds, in milliseconds.
my_uncrossed_threshold_ms = 45   # example data: smallest reliable gap, hands uncrossed
my_crossed_threshold_ms = 140    # example data: smallest reliable gap, hands crossed

ratio = my_crossed_threshold_ms / my_uncrossed_threshold_ms
print(f"Your uncrossed threshold: {my_uncrossed_threshold_ms} ms")
print(f"Your crossed threshold:   {my_crossed_threshold_ms} ms")
print(f"Ratio (crossed / uncrossed): {ratio:.1f}x")
print("Shore et al. (2002) reported thresholds of 34 ms (uncrossed) and 124 ms (crossed), a ratio of about")
print("3.6x. This is a rough, informal estimate from a handful of trials, not a controlled psychophysics")
print("experiment, so treat any similarity to the published ratio as suggestive rather than a replication.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-54.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")